# Exploración inicial del dataset Monster

Notebook para echar un primer vistazo a las fotos capturadas: cuántas hay, cómo se ven, distribución de tamaños, etc.

**Uso:**
- Ejecutar después de capturar fotos y renombrarlas con `scripts/rename_photos.py`.
- Sirve para detectar problemas pronto (fotos demasiado oscuras, mal encuadradas, etc.).

In [ ]:
# Imports
import sys
from pathlib import Path

# Permite importar desde src/ aunque el notebook esté en notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from src.utils.io_utils import load_image, list_images

# Configuración
DATA_RAW = Path('../data/raw')
plt.rcParams['figure.dpi'] = 90

## 1. Inventario por variante

In [ ]:
inventory = {}
for variante_dir in sorted(DATA_RAW.iterdir()):
    if variante_dir.is_dir():
        images = list_images(variante_dir)
        inventory[variante_dir.name] = images
        print(f'{variante_dir.name:20s}  {len(images):4d} fotos')

total = sum(len(v) for v in inventory.values())
print(f'\nTOTAL: {total} fotos')

## 2. Visualizar una muestra de cada variante

In [ ]:
n_per_variant = 4
n_variants = len(inventory)

if n_variants == 0:
    print('Aún no hay fotos. Captura algunas y ejecuta esta celda otra vez.')
else:
    fig, axes = plt.subplots(n_variants, n_per_variant,
                              figsize=(3 * n_per_variant, 3 * n_variants))
    if n_variants == 1:
        axes = axes.reshape(1, -1)

    for row, (variante, images) in enumerate(inventory.items()):
        sample = images[:n_per_variant] if len(images) >= n_per_variant else images
        for col in range(n_per_variant):
            ax = axes[row, col]
            if col < len(sample):
                img = load_image(sample[col])
                ax.imshow(img)
                ax.set_title(sample[col].name, fontsize=8)
            ax.axis('off')
        axes[row, 0].set_ylabel(variante, fontsize=11, rotation=0,
                                 ha='right', va='center')

    plt.tight_layout()
    plt.show()

## 3. Distribución de tamaños

Para detectar fotos anómalas (resoluciones raras, móviles distintos).

In [ ]:
sizes = []
for images in inventory.values():
    for path in images:
        img = load_image(path)
        h, w = img.shape[:2]
        sizes.append((w, h))

if sizes:
    widths, heights = zip(*sizes)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].hist(widths, bins=20, color='steelblue', edgecolor='black')
    axes[0].set_title('Anchos (px)')
    axes[1].hist(heights, bins=20, color='coral', edgecolor='black')
    axes[1].set_title('Altos (px)')
    plt.tight_layout()
    plt.show()
    print(f'Ancho:  min={min(widths)}, max={max(widths)}, media={np.mean(widths):.0f}')
    print(f'Alto:   min={min(heights)}, max={max(heights)}, media={np.mean(heights):.0f}')

## 4. Próximos pasos

Una vez que el dataset crezca lo suficiente:

1. **F2**: pasar a `notebooks/02_preprocesado.ipynb` y empezar a probar CLAHE.
2. **F4**: anotar bounding boxes con LabelImg o CVAT.
3. **Continuar** según el plan del README.